In [3]:
import numpy as np, pandas as pd, lightgbm as lgb, joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ----------------------------- config -----------------------------
CSV_PATH   = "aqi_processed_data.csv"
TARGET     = "pm2_5_ugm3"
HORIZONS   = range(1, 25)            # 1..24 hours ahead
TRAIN_END  = pd.Timestamp("2024-06-01")
VAL_END    = pd.Timestamp("2025-01-01")   # test = everything after this
GAP        = pd.Timedelta(hours=24)       # avoids target leakage across splits
USE_LOG    = True
PEAK_THR   = 90                      # ug/m3, "poor" and above (CPCB)
CPCB_BINS  = [30, 60, 90, 120, 250]  # PM2.5 category edges

# ----------------------------- load -------------------------------
df = pd.read_csv(CSV_PATH, parse_dates=["datetime"]).sort_values("datetime")
df = df.drop_duplicates("datetime").set_index("datetime")

# make sure the hourly index is continuous
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="h")
print("missing hours:", len(full_idx) - len(df))
df = df.reindex(full_idx).interpolate(limit=3)      # fill only short gaps
df[["festival_period", "crop_burning_season"]] = df[["festival_period", "crop_burning_season"]].ffill()
df.index.name = "datetime"
df = df.reset_index()
print("range:", df.datetime.min(), "->", df.datetime.max())

# ------------------------ feature engineering ----------------------
def build_features(d):
    d = d.copy()
    pm = d[TARGET]

    # wind components (direction is circular)
    rad = np.deg2rad(d["wind_dir_10m"])
    d["wind_u"] = -d["wind_speed_10m_kmh"] * np.sin(rad)
    d["wind_v"] = -d["wind_speed_10m_kmh"] * np.cos(rad)

    # cyclical time at the forecast origin
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    doy = d["datetime"].dt.dayofyear
    d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    # lags of PM2.5 and PM10
    for col in [TARGET, "pm10_ugm3"]:
        for l in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
            d[f"{col}_lag{l}"] = d[col].shift(l)

    # lags of other pollutants
    for col in ["no2_ugm3", "co_ugm3", "so2_ugm3", "o3_ugm3"]:
        for l in [3, 24]:
            d[f"{col}_lag{l}"] = d[col].shift(l)

    # rolling PM2.5 statistics (include current hour, known at origin)
    for w in [6, 24, 72, 168]:
        d[f"pm_roll{w}_mean"] = pm.rolling(w).mean()
    d["pm_roll24_max"] = pm.rolling(24).max()
    d["pm_roll72_max"] = pm.rolling(72).max()
    d["pm_roll24_std"] = pm.rolling(24).std()
    d["pm_trend_24"]   = pm - pm.shift(24)
    d["pm_trend_6_24"] = d["pm_roll6_mean"] - d["pm_roll24_mean"]

    # meteorology history
    d["wind_roll24_mean"]  = d["wind_speed_10m_kmh"].rolling(24).mean()
    d["stagnation_24h"]    = d["wind_stagnation"].rolling(24).sum()
    d["humidity_roll24"]   = d["humidity_percent"].rolling(24).mean()
    d["temp_roll24"]       = d["temp_2m_c"].rolling(24).mean()
    d["solar_roll24"]      = d["solar_radiation_wm2"].rolling(24).mean()
    d["rain_6h"]           = d["precipitation_mm"].rolling(6).sum()
    d["rain_24h"]          = d["precipitation_mm"].rolling(24).sum()
    d["rain_72h"]          = d["precipitation_mm"].rolling(72).sum()
    d["pressure_diff_3h"]  = d["pressure_msl_hpa"].diff(3)
    d["pressure_diff_24h"] = d["pressure_msl_hpa"].diff(24)
    d["aod_roll24"]        = d["aod"].rolling(24).mean()
    d["dust_roll24"]       = d["dust_ugm3"].rolling(24).mean()

    # how long the festival / burning flags have been active recently
    d["festival_7d"] = d["festival_period"].rolling(168).sum()
    d["burning_7d"]  = d["crop_burning_season"].rolling(168).sum()
    return d

df = build_features(df)

DROP = ["datetime", "year", "wind_dir_10m"]
BASE_FEATS = [c for c in df.columns if c not in DROP]

def make_xy(d, h):
    """Features at origin t, plus calendar info for the target time t+h."""
    X = d[BASE_FEATS].copy()
    ts = d["datetime"] + pd.Timedelta(hours=h)
    X["target_hour"]  = ts.dt.hour
    X["target_dow"]   = ts.dt.dayofweek
    X["target_month"] = ts.dt.month
    X["target_festival"] = d["festival_period"].shift(-h)
    X["target_burning"]  = d["crop_burning_season"].shift(-h)
    y = d[TARGET].shift(-h)
    return X, y

# ----------------------------- training ---------------------------
params = dict(
    objective="regression", n_estimators=3000, learning_rate=0.03,
    num_leaves=31, min_child_samples=40, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1,
)

fwd = (lambda y: np.log1p(y)) if USE_LOG else (lambda y: y)
inv = (lambda y: np.expm1(y)) if USE_LOG else (lambda y: y)

def cpcb_cat(x):
    return np.digitize(x, CPCB_BINS)

models, best_iters, rows, preds_store = {}, {}, [], {}

for h in HORIZONS:
    X, y = make_xy(df, h)
    valid = X.notna().all(axis=1) & y.notna()
    t = df["datetime"]

    tr = valid & (t < TRAIN_END - GAP)
    va = valid & (t >= TRAIN_END) & (t < VAL_END - GAP)
    te = valid & (t >= VAL_END)

    m = lgb.LGBMRegressor(**params)
    m.fit(X[tr], fwd(y[tr]),
          eval_set=[(X[va], fwd(y[va]))],
          eval_metric="l1",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    models[h], best_iters[h] = m, m.best_iteration_

    p = np.clip(inv(m.predict(X[te])), 0, None)
    yt = y[te].values
    base_yday = df[TARGET].shift(24 - h)[te].values   # same hour yesterday
    base_now  = df[TARGET][te].values                 # persistence
    preds_store[h] = (yt, p, base_yday)

    peak = yt > PEAK_THR
    rows.append(dict(
        h=h, iters=m.best_iteration_,
        MAE=mean_absolute_error(yt, p),
        RMSE=np.sqrt(mean_squared_error(yt, p)),
        MAE_yesterday=mean_absolute_error(yt, base_yday),
        MAE_persist=mean_absolute_error(yt, base_now),
        MAE_peaks=mean_absolute_error(yt[peak], p[peak]) if peak.any() else np.nan,
        cat_acc=(cpcb_cat(yt) == cpcb_cat(p)).mean(),
    ))
    print(f"h={h:2d} done | MAE={rows[-1]['MAE']:.2f} | yday-baseline MAE={rows[-1]['MAE_yesterday']:.2f}")

res = pd.DataFrame(rows).set_index("h")
print(res.round(2))
print("\nAverage over horizons:\n", res.mean().round(3))

# ----------------------- overall test summary ---------------------
yt_all = np.concatenate([v[0] for v in preds_store.values()])
p_all  = np.concatenate([v[1] for v in preds_store.values()])
b_all  = np.concatenate([v[2] for v in preds_store.values()])
pk = yt_all > PEAK_THR
print(f"\nOverall MAE  model={mean_absolute_error(yt_all, p_all):.2f} "
      f"| same-hour-yesterday={mean_absolute_error(yt_all, b_all):.2f}")
print(f"Peak-hour MAE (> {PEAK_THR}): model={mean_absolute_error(yt_all[pk], p_all[pk]):.2f} "
      f"| yesterday={mean_absolute_error(yt_all[pk], b_all[pk]):.2f}")
print(f"Peak bias (pred - true): {np.mean(p_all[pk] - yt_all[pk]):.1f}")

# ---------------------- feature importance ------------------------
for h in [1, 12, 24]:
    imp = pd.Series(models[h].booster_.feature_importance("gain"),
                    index=models[h].booster_.feature_name()).sort_values(ascending=False)
    print(f"\nTop features, h={h}:\n", (imp / imp.sum()).head(12).round(3))

# -------- final refit on train + validation, then save ------------
final_models = {}
for h in HORIZONS:
    X, y = make_xy(df, h)
    valid = X.notna().all(axis=1) & y.notna() & (df["datetime"] < VAL_END - GAP)
    p2 = {**params, "n_estimators": int(best_iters[h] * 1.1)}
    final_models[h] = lgb.LGBMRegressor(**p2).fit(X[valid], fwd(y[valid]))

joblib.dump({"models": final_models, "features": BASE_FEATS}, "pm25_24h_lgbm.joblib")

# ------------------- forecast the next 24 hours -------------------
def forecast_next_24h(history, future_flags=None):
    """
    history: DataFrame with the same raw columns as your CSV, hourly, at least ~200 rows,
             ending at the hour you are forecasting from.
    future_flags: optional DataFrame of 24 rows with columns
             festival_period, crop_burning_season for the next 24 hours.
             If None, the last observed values are carried forward.
    """
    d = build_features(history.sort_values("datetime").reset_index(drop=True))
    last = d.iloc[[-1]]
    out = []
    for h in HORIZONS:
        X = last[BASE_FEATS].copy()
        ts = last["datetime"].iloc[0] + pd.Timedelta(hours=h)
        X["target_hour"], X["target_dow"], X["target_month"] = ts.hour, ts.dayofweek, ts.month
        if future_flags is not None:
            X["target_festival"] = future_flags["festival_period"].iloc[h - 1]
            X["target_burning"]  = future_flags["crop_burning_season"].iloc[h - 1]
        else:
            X["target_festival"] = last["festival_period"].iloc[0]
            X["target_burning"]  = last["crop_burning_season"].iloc[0]
        pred = float(np.clip(inv(final_models[h].predict(X))[0], 0, None))
        out.append((ts, pred))
    return pd.DataFrame(out, columns=["datetime", "pm2_5_forecast"])

# example: forecast from the last hour in your data
print(forecast_next_24h(pd.read_csv(CSV_PATH, parse_dates=["datetime"]).tail(400)))

missing hours: 0
range: 2022-08-05 00:00:00 -> 2025-11-26 23:00:00


C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


h= 1 done | MAE=1.88 | yday-baseline MAE=7.83


C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


h= 2 done | MAE=2.95 | yday-baseline MAE=7.83


...

h=22 done | MAE=7.46 | yday-baseline MAE=7.82


C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


h=23 done | MAE=7.38 | yday-baseline MAE=7.82


C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


h=24 done | MAE=7.54 | yday-baseline MAE=7.81
    iters   MAE   RMSE  MAE_yesterday  MAE_persist  MAE_peaks  cat_acc
h                                                                     
1     485  1.88   3.21           7.83         2.04       9.74     0.94
2     565  2.95   4.71           7.83         3.67      13.11     0.91
3     560  3.90   6.01           7.82         5.00      15.62     0.88
4     658  4.63   7.03           7.82         6.07      16.72     0.86
5     361  5.19   7.93           7.82         6.94      18.08     0.85
6     285  5.52   8.32           7.82         7.63      19.62     0.83
7     263  5.87   8.92           7.82         8.17      20.56     0.82
8     238  6.18   9.41           7.82         8.59      24.15     0.81
9     190  6.34   9.58           7.82         8.91      24.93     0.81
10    214  6.49   9.84           7.82         9.14      26.49     0.81
11    199  6.62  10.11           7.82         9.30      26.85     0.80
12    206  6.66  10.22         

In [2]:
pip install lightgbm

  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.4 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.7 MB/s eta 0:00:00
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Users\Vishal Panindre\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
